In [1]:
from google.colab import drive
drive.mount('/content/drive')

# install all required packages
!pip install -q torch torchvision timm transformers \
    accelerate pillow pandas numpy scikit-learn \
    matplotlib tqdm huggingface_hub

Mounted at /content/drive


In [2]:
import os

if not os.path.exists("/content/XrVLM"):
    !git clone https://github.com/Tabassum-Sumaiya13/XrVLM.git /content/XrVLM
    print("✅ Cloned")
else:
    !cd /content/XrVLM && git pull
    print("✅ Repo up to date")

os.chdir("/content/XrVLM")

Cloning into '/content/XrVLM'...
remote: Enumerating objects: 24, done.
remote: Counting objects: 100% (24/24), done.
remote: Compressing objects: 100% (17/17), done.
remote: Total 24 (delta 6), reused 24 (delta 6), pack-reused 0 (from 0)
Receiving objects: 100% (24/24), 38.44 KiB | 1.48 MiB/s, done.
Resolving deltas: 100% (6/6), done.
✅ Cloned


In [3]:
import os, shutil
from pathlib import Path

repo  = Path("/content/XrVLM")
drive = Path("/content/drive/MyDrive/XrVLM ")  # trailing space — your Drive folder name

# ── data/raw ─────────────────────────────────────────────────
repo_raw = repo / "data/raw"
if repo_raw.is_symlink(): repo_raw.unlink()
elif repo_raw.exists(): shutil.rmtree(repo_raw)
(repo / "data").mkdir(exist_ok=True)
os.symlink(drive / "data/raw/sample", repo_raw)

# ── outputs ──────────────────────────────────────────────────
repo_out = repo / "outputs"
if repo_out.is_symlink(): repo_out.unlink()
elif repo_out.exists(): shutil.rmtree(repo_out)
os.symlink(drive / "outputs", repo_out)

# ── stray data/sample dir ────────────────────────────────────
stray = repo / "data/sample"
if stray.exists() and not stray.is_symlink():
    shutil.rmtree(stray)

# ── verify ───────────────────────────────────────────────────
images = list((repo_raw / "images").glob("*.png"))
print(f"Images  : {len(images)}")          # expect 5606
print(f"CSV     : {(repo_raw/'sample_labels.csv').exists()}")   # expect True
print(f"Outputs : {(repo/'outputs/checkpoints').exists()}")     # expect True

Images  : 5606
CSV     : True
Outputs : True


In [4]:
with open("/content/XrVLM/dataset.py") as f:
    src = f.read()

# Fix 1: CSV filename
src = src.replace(
    'csv_path = raw_dir / "Data_Entry_2017.csv"\n    if not csv_path.exists():\n        # Try alternate common name\n        csv_path = raw_dir / "Data_Entry_2017_v2020.csv"',
    'csv_path = raw_dir / "sample_labels.csv"\n    if not csv_path.exists():\n        csv_path = raw_dir / "Data_Entry_2017.csv"\n    if not csv_path.exists():\n        csv_path = raw_dir / "Data_Entry_2017_v2020.csv"'
)

# Fix 2: images/ subfolder support
src = src.replace(
    '    # Flat layout fallback: *.png directly in raw_dir\n    if not index:\n        for p in raw_dir.iterdir():\n            if p.suffix.lower() == ".png":\n                index[p.name] = p',
    '    # images/ subfolder (NIH sample layout)\n    if not index:\n        images_subdir = raw_dir / "images"\n        if images_subdir.exists():\n            for p in images_subdir.iterdir():\n                if p.suffix.lower() == ".png":\n                    index[p.name] = p\n    # Flat fallback\n    if not index:\n        for p in raw_dir.iterdir():\n            if p.suffix.lower() == ".png":\n                index[p.name] = p'
)

with open("/content/XrVLM/dataset.py", "w") as f:
    f.write(src)

print("CSV patch    :", "sample_labels.csv" in src)
print("Images patch :", "images_subdir" in src)

CSV patch    : True
Images patch : True


In [5]:
from huggingface_hub import snapshot_download
from pathlib import Path

model_dir = "/content/drive/MyDrive/XrVLM /models/chexagent"
Path(model_dir).mkdir(parents=True, exist_ok=True)

if len(list(Path(model_dir).glob("*.json"))) < 3:
    print("Downloading CheXagent-2-3b (~6GB) to Drive...")
    snapshot_download(
        repo_id="StanfordAIMI/CheXagent-2-3b",
        local_dir=model_dir,
        ignore_patterns=["*.msgpack", "*.h5"],
    )
    print("✅ Model saved to Drive")
else:
    print(f"✅ Already on Drive ({len(list(Path(model_dir).iterdir()))} files)")with open("/content/XrVLM/config.py") as f:
    src = f.read()

src = src.replace(
    'VLM_MODEL_ID = "StanfordAIMI/CheXagent-2-3b"',
    'VLM_MODEL_ID = "/content/drive/MyDrive/XrVLM /models/chexagent"'
)

with open("/content/XrVLM/config.py", "w") as f:
    f.write(src)

print("✅ VLM path updated:",
      "chexagent" in open("/content/XrVLM/config.py").read())

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Fetching 17 files:   0%|          | 0/17 [00:00<?, ?it/s]

✅ Model saved to Drive


In [6]:
with open("/content/XrVLM/config.py") as f:
    src = f.read()

src = src.replace(
    'VLM_MODEL_ID = "StanfordAIMI/CheXagent-2-3b"',
    'VLM_MODEL_ID = "/content/drive/MyDrive/XrVLM /models/chexagent"'
)

with open("/content/XrVLM/config.py", "w") as f:
    f.write(src)

print("✅ VLM path updated:",
      "chexagent" in open("/content/XrVLM/config.py").read())

✅ VLM path updated: False


In [7]:
with open("/content/XrVLM/config.py") as f:
    src = f.read()

# Find current VLM_MODEL_ID line — whatever it is
import re
current = re.search(r'VLM_MODEL_ID\s*=\s*["\'](.+?)["\']', src)
print("Current VLM_MODEL_ID:", current.group(1) if current else "NOT FOUND")

# Replace it regardless of current value
src = re.sub(
    r'VLM_MODEL_ID\s*=\s*["\'].+?["\']',
    'VLM_MODEL_ID = "/content/drive/MyDrive/XrVLM /models/chexagent"',
    src
)

with open("/content/XrVLM/config.py", "w") as f:
    f.write(src)

# Verify
current_after = re.search(r'VLM_MODEL_ID\s*=\s*["\'](.+?)["\']', src)
print("Updated VLM_MODEL_ID:", current_after.group(1) if current_after else "NOT FOUND")

Current VLM_MODEL_ID: StanfordAIMI/CheXagent-2-3b
Updated VLM_MODEL_ID: /content/drive/MyDrive/XrVLM /models/chexagent


In [ ]:
import os
os.chdir("/content/XrVLM")

!python train.py \
    --epochs-stage1 15 \
    --epochs-stage2 0 \
    --batch 16 \
    2>&1 | tee outputs/train_log.txt

In [ ]:
import os
os.chdir("/content/XrVLM")

# Stage 1 screens all images → low-confidence → real CheXagent VLM
!python pipeline.py data/raw/images \
    --threshold 0.65 \
    2>&1 | tee outputs/pipeline_log.txt

In [ ]:
import json, pandas as pd
from pathlib import Path

log_lines = Path("/content/XrVLM/outputs/pipeline_log.txt").read_text().splitlines()

# Print last 30 lines (batch summary)
print("\n".join(log_lines[-30:]))

# Parse any JSON result lines into a DataFrame
records = []
for line in log_lines:
    if line.strip().startswith("{"):
        try: records.append(json.loads(line.strip()))
        except: pass

if records:
    df = pd.DataFrame(records)
    print(f"\n✅ {len(df)} results parsed")
    for col in ["final_finding", "needs_radiologist_review",
                "conflict_detected", "routed_to_stage2"]:
        if col in df.columns:
            print(f"\n{col}:\n{df[col].value_counts()}")
    df.to_csv("outputs/all_results.csv", index=False)
    print("\n✅ Saved: outputs/all_results.csv → also on Drive")